In [1]:
import heapq
import os
from collections import Counter

In [2]:
# Main
input_file = "sample_text_for_hf_text_compressor.txt"
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()
compressed_file = "compressed.bin"
decompressed_file = "restored_output.txt"


In [8]:
# Read file
try:
    with open(input_file, "r") as f:
        text = f.read()
except FileNotFoundError:
    print(f"Error: File '{input_file}' not found.")
    exit(1)

# Build Tree & Codes
freq_table = freq_counter(text)
hf_tree_root = build_hf_tree(freq_table)
codes = get_codes(hf_tree_root)

# Compress
compressed_data = compress_text(text, codes)
with open(compressed_file, "wb") as f:
    f.write(compressed_data)
print(f"compressed file size: {os.path.getsize(compressed_file)} bytes")

# Decompress
decompressed_text = decompress_bytes(compressed_data, hf_tree_root)
with open(decompressed_file, "w") as f:
    f.write(decompressed_text)
print(f"decompressed file size: {os.path.getsize(decompressed_file)} bytes")

# compression statistics
print("\n --- COMPRESSIOIN STATISTICS ---")
if text == decompressed_text:
    compression_ratio = (os.path.getsize(input_file) - os.path.getsize(compressed_file)) / os.path.getsize(input_file) * 100
    compression_factor = os.path.getsize(input_file) / os.path.getsize(compressed_file)
    print(f"Data Shrunk by: {compression_ratio:.2f}%")
    print(f"Compression Factor: {compression_factor:.2f}x")
    print("Decompression successful! Original and decompressed texts match.")
else:
    print("Decompression failed! Original and decompressed texts do not match.")

compressed file size: 317 bytes
decompressed file size: 568 bytes

 --- COMPRESSIOIN STATISTICS ---
Data Shrunk by: 44.19%
Compression Factor: 1.79x
Decompression successful! Original and decompressed texts match.


In [7]:
class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left =None
        self.right = None
    # special method to compare nodes in heapq based on frequency
    def __lt__(self, other):
        return self.freq < other.freq


def freq_counter(text):
    return Counter(text)
print("Frequency Table: ", freq_counter(text))


def build_hf_tree(freq_table):
    heap = []
    for char, freq in freq_table.items():
        heapq.heappush(heap, Node(char, freq)) # creat initial nodes
        # iteratively travel bottom to top while merging lowest two nodes
    while len(heap)>1:   
        first = heapq.heappop(heap) # get 1st smallest node
        second = heapq.heappop(heap) # get 2nd smallest node
        
        merged = Node(None, first.freq + second.freq)
        merged.left = first # assign left child
        merged.right = second # assign right child
        
        heapq.heappush(heap, merged) # # push the new parent back into the heap
    return heap[0]


# generate the hf codes using hf tree we just built        
def get_codes(node, current_code="", codes=None):
    if codes is None:  # initialize code dict in first call
        codes = {}
    if node is None: # base case: if node is empty, return current code dict
        return codes
    # leaf node is reached - assign current code to char
    if node.char is not None: 
        codes[node.char] = current_code
        return codes
        
    # recursive calls
    get_codes(node.left, current_code+"0", codes)
    get_codes(node.right, current_code+"1", codes)
    return codes
print("\nHuffman Codes: ", get_codes(hf_tree_root))


def compress_text(text, codes):
    # text characters --> string of bits
    bit_str = "".join(codes[ch] for ch in text) # encoded the text
    
    # string of bits --> padded string of bits
    padded_amt = 8 - (len(bit_str) % 8)
    if padded_amt == 8:
        padded_amt = 0
    bit_str += "0" * padded_amt
    # store the padding amount as the very first byte
    compressed_bytes = bytearray()
    compressed_bytes.append(padded_amt)
    
    # string of bits --> 1 byte (8 bit) chunks
    for i in range(0, len(bit_str), 8):
        byte_chunk = bit_str[i:i+8]
        compressed_bytes.append(int(byte_chunk, 2))
    return bytes(compressed_bytes) # convert bytearray to immutable bytes object so streamlit won't complain


def decompress_bytes(compressed_bytes, root):
    if not root:
        return ""
    padded_amt = compressed_bytes[0]
    
    # 1 byte (8 bit) chunks --> padded string of bits
    bit_str = ""
    for byte in compressed_bytes[1:]:
        bit_str += f"{byte:08b}"
    
    # padded string of bits --> string of bits
    if padded_amt>0:
        bit_str = bit_str[:-padded_amt]
    
    # string of bits --> text characters
    decoded_text = ""
    current_node = root
    # traverse the hf tree from root to leaf according to the bits in bit_str
    for bit in bit_str:
        if bit == "0":
            current_node = current_node.left
        else:
            current_node = current_node.right
        
        if current_node.char is not None: # leaf node reached
            decoded_text += current_node.char
            current_node = root # reset to root for next char
    return decoded_text


Frequency Table:  Counter({'~': 160, ' ': 43, 'e': 38, 'H': 32, 'p': 29, 'G': 16, 'r': 15, 'i': 14, 'c': 13, 'k': 13, 'F': 12, 'E': 11, '0': 10, 'd': 9, '9': 9, 'P': 9, 'o': 8, '8': 8, '7': 7, '\n': 6, 'f': 6, 's': 6, 't': 6, '5': 6, '6': 6, 'l': 5, 'A': 4, 'D': 4, 'h': 4, '4': 4, 'C': 3, 'a': 3, '.': 3, '1': 3, '2': 3, '3': 3, 'S': 3, 'B': 2, 'T': 2, 'u': 2, 'w': 2, 'R': 2, '_': 2, 'q': 1, 'b': 1, 'n': 1, 'x': 1, 'j': 1, 'm': 1, 'v': 1, 'z': 1, 'y': 1, 'g': 1, 'I': 1, ',': 1, '?': 1, 'W': 1, 'O': 1, '!': 1})

Huffman Codes:  {'2': '0000000', '3': '0000001', 'C': '0000010', 'S': '0000011', '.': '0000100', '1': '0000101', '\n': '000011', 'k': '00010', 'c': '00011', 'i': '00100', 'a': '0010100', 'u': '00101010', 'q': '001010110', 'x': '001010111', '7': '001011', 'p': '0011', 'r': '01000', '4': '0100100', ',': '010010100', '?': '010010101', 'n': '010010110', 'z': '010010111', 'v': '010011000', 'W': '010011001', 'O': '010011010', '!': '010011011', 'h': '0100111', 'H': '0101', 'G': '01100',

In [5]:
def hf_char_codes(node, prefix=""):
    if node is not None:
        if node.char is not None:
            print(f"'{node.char}': {prefix} (freq: {node.freq})")
        hf_char_codes(node.left, prefix + "0")
        hf_char_codes(node.right, prefix + "1")
# frequent characters get shorter codes
print("\n--- Huffman Tree Codes ---\n")
for char, code in sorted(codes.items()):
    print(f"'{char}'  -->  {code} | freq = {freq_table[char]}")


--- Huffman Tree Codes ---

'
'  -->  000011 | freq = 6
' '  -->  1110 | freq = 43
'!'  -->  010011011 | freq = 1
','  -->  010010100 | freq = 1
'.'  -->  0000100 | freq = 3
'0'  -->  110110 | freq = 10
'1'  -->  0000101 | freq = 3
'2'  -->  0000000 | freq = 3
'3'  -->  0000001 | freq = 3
'4'  -->  0100100 | freq = 4
'5'  -->  1111101 | freq = 6
'6'  -->  1111111 | freq = 6
'7'  -->  001011 | freq = 7
'8'  -->  011010 | freq = 8
'9'  -->  110100 | freq = 9
'?'  -->  010010101 | freq = 1
'A'  -->  0110111 | freq = 4
'B'  -->  01110101 | freq = 2
'C'  -->  0000010 | freq = 3
'D'  -->  0111100 | freq = 4
'E'  -->  110111 | freq = 11
'F'  -->  111101 | freq = 12
'G'  -->  01100 | freq = 16
'H'  -->  0101 | freq = 32
'I'  -->  011011010 | freq = 1
'O'  -->  010011010 | freq = 1
'P'  -->  011111 | freq = 9
'R'  -->  01110110 | freq = 2
'S'  -->  0000011 | freq = 3
'T'  -->  01101100 | freq = 2
'W'  -->  010011001 | freq = 1
'_'  -->  01110100 | freq = 2
'a'  -->  0010100 | freq = 3
'b'  -->

In [6]:
def print_tree_structure(node, level=0, prefix="Root: "):
    if node is None:
        return

    indent = "    " * level
    if node.char is not None:
        print(f"{indent}{prefix}[Char: {repr(node.char)} | Freq: {node.freq}]")
    else:
        print(f"{indent}{prefix}(Internal Freq: {node.freq})")

    print_tree_structure(node.left, level + 1, prefix="0 -> ")
    print_tree_structure(node.right, level + 1, prefix="1 -> ")

print("\n--- Visual Huffman Tree Structure ---\n") 
print_tree_structure(hf_tree_root)


--- Visual Huffman Tree Structure ---

Root: (Internal Freq: 562)
    0 -> (Internal Freq: 235)
        0 -> (Internal Freq: 107)
            0 -> (Internal Freq: 50)
                0 -> (Internal Freq: 24)
                    0 -> (Internal Freq: 12)
                        0 -> (Internal Freq: 6)
                            0 -> [Char: '2' | Freq: 3]
                            1 -> [Char: '3' | Freq: 3]
                        1 -> (Internal Freq: 6)
                            0 -> [Char: 'C' | Freq: 3]
                            1 -> [Char: 'S' | Freq: 3]
                    1 -> (Internal Freq: 12)
                        0 -> (Internal Freq: 6)
                            0 -> [Char: '.' | Freq: 3]
                            1 -> [Char: '1' | Freq: 3]
                        1 -> [Char: '\n' | Freq: 6]
                1 -> (Internal Freq: 26)
                    0 -> [Char: 'k' | Freq: 13]
                    1 -> [Char: 'c' | Freq: 13]
            1 -> (Internal Freq: 57)
 